## Step1:Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from delta.tables import *
from delta.tables import DeltaTable


##Step2: Generate Fact Table 

###2.1:Read Silver Orders

In [0]:
silver_orders = spark.table("silver_orders")

###2.2:Create Dimension Table(customer)

In [0]:
dim_customer = spark.table("dim_customers_scd2")\
    .filter("is_current = true")


###2.2:Create Dimension Table(product)

In [0]:
dim_product = spark.table("dim_products_scd2")\
    .filter("is_current = true")


###2.2:Create Dimension Table(Store)

In [0]:
dim_store = spark.table("silver_stores")

##2.3:Join orders table and customers

In [0]:
fact = silver_orders.alias("o")\
    .join(
        dim_customer.alias("c"),
        "customer_id",
        "left"
    )


##2.4:Join orders table and stores

In [0]:
fact = fact.alias("f")\
    .join(
        dim_store.alias("s"),
        "store_id",
        "left"
    )


##2.5:Join orders table and products

In [0]:

fact_with_product = fact.alias("f")\
    .join(
        dim_product.alias("p"),
        "product_id",
        "left"
    )



###2.6:Column selection in Fact Table

In [0]:
fact_orders = fact_with_product.select(
    "order_id",
    "order_ts",
    "customer_sk",
    col("p.product_sk").alias("product_sk"),
    "store_id",
    "quantity",
    col("f.unit_price").alias("unit_price"),
    "discount_pct",
    "gross_amount",
    "payment_method",
    "order_status"
)

fact_orders = fact_with_product.select(
    "order_id",
    "order_ts",
    "customer_sk",
    col("p.product_sk").alias("product_sk"),
    "store_id",
    "quantity",
    col("f.unit_price").alias("unit_price"),
    "discount_pct",
    "gross_amount",
    "payment_method",
    "order_status"
)

## Step3:Save Fact Table

In [0]:
fact_orders.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fact_orders")

In [0]:
print("No of records in fact table",fact_orders.count())

No of records in fact table 11498


In [0]:
display(fact_orders.limit(10))

order_id,order_ts,customer_sk,product_sk,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status
O0000001,2025-12-15T04:49:00.000Z,1651,623,S003,4,30260.29,0.15,121041.16,CARD,cancelled
O0000002,2026-03-13T03:29:00.000Z,706,null,S028,4,30049.26,0.1,102167.48,CARD,returned
O0000003,2026-03-05T07:20:00.000Z,2109,850,S072,5,59685.88,0.0,283507.93,COD,delivered
O0000004,2025-12-28T13:56:00.000Z,1383,29,S071,1,18705.7,0.05,15899.84,NETBANKING,returned
O0000005,2025-12-31T17:43:00.000Z,380,935,S049,2,46727.14,0.15,84108.85,COD,cancelled
O0000006,2026-03-11T02:59:00.000Z,2555,98,S019,2,12387.77,0.05,21059.21,COD,returned
O0000007,2025-12-26T03:16:00.000Z,1614,544,S020,2,54612.57,0.05,98302.63,COD,returned
O0000008,2025-12-09T20:57:00.000Z,1940,751,S029,1,29725.2,0.0,28238.94,UPI,returned
O0000009,2026-01-20T19:25:00.000Z,473,762,S050,4,84092.97,0.05,336371.88,UPI,delivered
O0000010,2026-03-21T06:52:00.000Z,195,65,S049,5,31466.94,0.0,157334.7,COD,delivered
